# S04 — Polynomial Regression Quest 🎯
### Ridge, Lasso and Alpha — beginner-friendly Colab tutorial
**8 missions • 100 XP • 85–100 minutes • CPU only**

**Learning outcomes:** Understand curved data, underfitting, overfitting, Ridge (L2), Lasso (L1), alpha and fair cross-validation. No external dataset or paid service required.

**Instructions:** Run cells in order. Write your answers in the spaces marked **YOUR ANSWER**. The test set is reserved until Mission 8. Save a personal copy via **File → Save a copy in Drive**. XP rewards reasoning and completed work, not speed or the lowest error.

## Before we start: the simplest explanation
A linear model draws a straight line. A polynomial model can draw a curve by adding features like x² and x³.

A model can be **too simple (underfitting)** or **too complicated (overfitting)**. A model that memorizes training noise may predict new examples poorly.

**Ridge (L2)** tells the model: *Fit the data, but avoid very large coefficients.* It generally shrinks coefficients without making them exactly zero.

**Lasso (L1)** tells the model: *Fit the data, and you may completely switch off some coefficients.* It can make some coefficients exactly zero. Zero does not prove a feature is useless.

**Alpha (α)** is the strength of the penalty. Small alpha = gentle brake; large alpha = stronger brake. **Too much alpha may cause underfitting.** Larger alpha is not always better.

**Why scale?** Ridge and Lasso penalize coefficient sizes. Polynomial columns may have very different scales. Put `StandardScaler` **inside the Pipeline**, so each cross-validation fold fits it using training data only.

**Our rule:** Choose models using development cross-validation, and use the reserved test only once at the end.

## Mission 1 — Think before coding (10 XP, 5 min)

| Model | Training MAE | Validation MAE |
|---|---:|---:|
| A | 12 | 13 |
| B | 4 | 5 |
| C | 1 | 18 |

**Questions:** Which model might underfit? Which may overfit? Which would you choose? Why is the smallest training error not enough?

**YOUR ANSWER:** *(double-click and type)*

## Mission 2 — Generate curved data (10 XP, 10 min)
We make 500 synthetic observations. Keep **100 for final testing**; use **400 for development**. The data contain a cubic pattern plus random noise.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEED = 42
rng = np.random.default_rng(SEED)
X = np.sort(rng.uniform(-3, 3, 500)).reshape(-1, 1)
y = 4 + 1.5*X[:, 0] - 1.2*X[:, 0]**2 + 0.35*X[:, 0]**3 + rng.normal(0, 3, 500)
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.20, random_state=SEED)
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
assert len(X_dev) == 400 and len(X_test) == 100
print("Development:", len(X_dev), "| Final test:", len(X_test))
plt.scatter(X_dev[:, 0], y_dev, alpha=.5, s=14)
plt.xlabel("Input x"); plt.ylabel("Target y"); plt.title("Synthetic curved data"); plt.show()

**Checkpoint:** You should see a noisy curve and 400/100 rows.

**YOUR ANSWER:** Is a straight line likely to fit perfectly? What is noise? Why keep 100 rows unseen?

## Mission 3 — Try polynomial degrees (15 XP, 15 min)
Degree 1 is a straight line. Degree 2 can curve. Higher degrees can bend more. **MAE** is the average absolute prediction error: smaller is better.

With **5-fold cross-validation**, each round trains on 320 development rows and validates on 80. All models use the same folds.

In [ ]:
def make_model(degree, penalty="none", alpha=1.0):
    if penalty == "ridge":
        estimator = Ridge(alpha=alpha)
    elif penalty == "lasso":
        estimator = Lasso(alpha=alpha, max_iter=100000)
    elif penalty == "none":
        estimator = LinearRegression()
    else:
        raise ValueError("Unknown penalty")
    return Pipeline([
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ("scale", StandardScaler()),
        ("model", estimator)
    ])

def evaluate_cv(model):
    scores = cross_validate(model, X_dev, y_dev, cv=cv,
                            scoring="neg_mean_absolute_error",
                            return_train_score=True)
    return {
        "train_mae": -scores["train_score"].mean(),
        "validation_mae": -scores["test_score"].mean(),
        "validation_sd": scores["test_score"].std(ddof=1)
    }

degree_rows = []
for degree in [1, 2, 3, 5, 8]:
    degree_rows.append({"model": f"Linear degree {degree}", "degree": degree,
                        "penalty": "none", "alpha": None,
                        **evaluate_cv(make_model(degree))})
degree_table = pd.DataFrame(degree_rows)
display(degree_table.round(3))

**YOUR ANSWER:** Which degree has the smallest training MAE? Which has the smallest validation MAE? Why can they differ? Is degree 8 always best?

## Mission 4 — Explain Ridge, Lasso and alpha (15 XP, 10 min)
**Ridge** uses an L2 penalty (sum of squared coefficients). **Lasso** uses an L1 penalty (sum of absolute coefficients). These are added to the fitting objective, with alpha controlling penalty strength. The intercept is normally not penalized.

**Fill the blanks:**
1. Ridge makes coefficients ______ but usually not exactly ______.
2. Lasso can set some coefficients exactly to ______.
3. Alpha controls penalty ______.
4. If alpha is too large, the model might ______.
5. We scale because ______.

**True or false?** A zero Lasso coefficient proves a feature is scientifically irrelevant. Explain.

## Mission 5 — Change alpha (20 XP, 15 min)
Keep polynomial degree **5** fixed. Compare Ridge and Lasso at the same alpha values. Changing one factor at a time makes the experiment easier to understand.

In [ ]:
regularisation_rows = []
for penalty in ["ridge", "lasso"]:
    for alpha in [0.01, 0.1, 1.0, 10.0, 100.0]:
        regularisation_rows.append({
            "model": f"{penalty} degree 5 alpha {alpha}",
            "degree": 5, "penalty": penalty, "alpha": alpha,
            **evaluate_cv(make_model(5, penalty, alpha))
        })
regularisation_table = pd.DataFrame(regularisation_rows)
display(regularisation_table.round(3))
for penalty in ["ridge", "lasso"]:
    part = regularisation_table[regularisation_table.penalty == penalty]
    plt.plot(part.alpha, part.validation_mae, marker="o", label=penalty)
plt.xscale("log"); plt.xlabel("Alpha (log scale)")
plt.ylabel("Mean validation MAE — lower is better")
plt.title("Effect of alpha on new-data predictions"); plt.legend(); plt.show()

**YOUR ANSWER:** For each method, which alpha gives the smallest validation MAE *among those tested*? Does increasing alpha always improve predictions? What happens when the penalty becomes too strong?

**Remember:** Polynomial degree changes available features. Alpha changes the strength of the coefficient penalty.

## Mission 6 — Inspect the coefficients (10 XP, 10 min)
Fit models on **development data only**. Compare coefficients. Lasso may set some to exactly zero; Ridge generally does not.

In [ ]:
for penalty in ["ridge", "lasso"]:
    model = make_model(5, penalty, alpha=1.0)
    model.fit(X_dev, y_dev)
    coefficients = model.named_steps["model"].coef_
    print(penalty, "coefficients:", np.round(coefficients, 4))
    print("Near zero:", np.sum(np.isclose(coefficients, 0, atol=1e-10)))

**YOUR ANSWER:** Which method can remove coefficients? Does a zero coefficient mean that x³ or x⁴ has no scientific relationship with y? (Hint: correlated features can substitute for each other.)

## Mission 7 — Select and LOCK your model (10 XP, 10 min)
Use **development cross-validation MAE only**. Do not look at the reserved test. The code picks the lowest validation MAE from our candidates. You may explain why a simpler model is preferable if candidates are practically tied.

In [ ]:
candidates = pd.concat([degree_table, regularisation_table], ignore_index=True)
ranked = candidates.sort_values("validation_mae").reset_index(drop=True)
display(ranked[["model", "train_mae", "validation_mae", "validation_sd"]].round(3))
chosen = ranked.iloc[0]
selected_degree = int(chosen.degree)
selected_penalty = str(chosen.penalty)
selected_alpha = 1.0 if pd.isna(chosen.alpha) else float(chosen.alpha)
print("LOCKED:", chosen.model)
print("Validation MAE:", round(chosen.validation_mae, 3))

**YOUR ANSWER:** Write the chosen degree, penalty, alpha and validation MAE. Explain your decision in two sentences. **Do not revise the choice after Mission 8.**

## Mission 8 — Final test ONCE (10 XP, 10 min)
Refit the locked model using all 400 development rows. Evaluate the reserved 100 test rows only now. The test is a final check, **not another model-selection tool**.

In [ ]:
assert selected_degree is not None
final_model = make_model(selected_degree, selected_penalty, selected_alpha)
final_model.fit(X_dev, y_dev)
predictions = final_model.predict(X_test)
print("Final test MAE:", round(mean_absolute_error(y_test, predictions), 3))
print("Final test RMSE:", round(np.sqrt(mean_squared_error(y_test, predictions)), 3))
residuals = y_test - predictions
plt.scatter(predictions, residuals, alpha=.6)
plt.axhline(0, color="black", linestyle="--")
plt.xlabel("Predicted y"); plt.ylabel("Residual: actual - predicted")
plt.title("Final test residuals"); plt.show()

**YOUR ANSWER:** Is test MAE close to validation MAE? What do the residuals show? Why must we not change alpha now?

## Exit ticket (answer in your own words)
1. Why might a straight line underfit curved data?
2. Why is the lowest training error not enough?
3. Explain Ridge versus Lasso to a friend.
4. What happens when alpha becomes too large?
5. Why is scaling inside the cross-validation pipeline important?

### Optional bonus (+20 XP)
**+10 XP:** Plot degree 1, 3 and 8 predictions across the observed x-range. Describe which is too simple or too wiggly.

**+10 XP:** Explain why random cross-validation may be inappropriate for time series or repeated records from one customer.

**Final reminder:** Synthetic data are for learning, not proof about real-world behavior. Save your own Colab copy; don't commit student answers or personal data to the public course repository.